In [1]:
import re
import json
from datetime import datetime
from pathlib import Path
import torch


In [2]:
VERIFIER_OUTPUT_PATH = "/kaggle/working/verifier_blackboard.pt"

print("Verifier output:", VERIFIER_OUTPUT_PATH)


Verifier output: /kaggle/working/verifier_blackboard.pt


In [3]:
# Load the Explanation Agent blackboard.
# The Explanation Agent saves:
# {
#   "agent": "ExplanationAgent",
#   "input_agent": "TriageAgent",
#   "output": explanation_output,
#   "timestamp": ...
# }
#
# Therefore the Verifier must unwrap the nested "output" field.

candidate_paths = [
    "/kaggle/working/explanation_blackboard.pt",
    "/kaggle/usr/lib/notebooks/premshaw23/"
    "explanation_agent/explanation_blackboard.pt",
]

EXPLANATION_BLACKBOARD_PATH = None

for candidate in candidate_paths:
    if Path(candidate).exists():
        EXPLANATION_BLACKBOARD_PATH = candidate
        break

if EXPLANATION_BLACKBOARD_PATH is None:
    raise FileNotFoundError(
        "Explanation blackboard not found. Checked:\n"
        + "\n".join(candidate_paths)
    )

raw_blackboard = torch.load(
    EXPLANATION_BLACKBOARD_PATH,
    map_location="cpu",
    weights_only=False
)

if not isinstance(raw_blackboard, dict):
    raise TypeError("Explanation blackboard must contain a dictionary.")

# Unwrap the Explanation Agent blackboard.
if isinstance(raw_blackboard.get("output"), dict):
    explanation_output = raw_blackboard["output"]
else:
    explanation_output = raw_blackboard

if not isinstance(explanation_output, dict):
    raise TypeError("Explanation output must be a dictionary.")

print("Loaded Explanation blackboard:")
print(EXPLANATION_BLACKBOARD_PATH)

print("\nBlackboard keys:")
print(list(raw_blackboard.keys()))

print("\nExplanation output keys:")
print(list(explanation_output.keys()))


Loaded Explanation blackboard:
/kaggle/usr/lib/notebooks/premshaw23/explanation_agent/explanation_blackboard.pt

Blackboard keys:
['agent', 'input_agent', 'output', 'timestamp']

Explanation output keys:
['agent', 'clinician_explanation', 'patient_explanation', 'claims', 'evidence', 'triage_provenance', 'grounding_policy']


In [4]:
# Consume the structured claims produced by the Explanation Agent directly.
# Do NOT reconstruct claims from clinician_explanation/patient_explanation.

claims = explanation_output.get("claims")
evidence = explanation_output.get("evidence", {})

if claims is None:
    raise KeyError(
        "Explanation Agent output does not contain 'claims'. "
        "Make sure the updated Explanation Agent notebook was run and "
        "its blackboard contains explanation_output['claims']."
    )

if not isinstance(claims, list):
    raise TypeError("explanation_output['claims'] must be a list.")

if not isinstance(evidence, dict):
    raise TypeError("explanation_output['evidence'] must be a dictionary.")

image_evidence = evidence.get("image", {})
clinical_evidence = evidence.get("clinical_note", {})
retrieved_evidence = evidence.get("retrieved", [])

validated_claims = []

for index, item in enumerate(claims):
    if not isinstance(item, dict):
        raise TypeError(f"Claim {index} is not a dictionary.")

    claim_text = item.get("claim")
    output_type = item.get("output", "unknown")

    if not isinstance(claim_text, str) or not claim_text.strip():
        raise ValueError(f"Claim {index} does not contain valid text.")

    validated_claims.append({
        **item,
        "claim": claim_text.strip(),
        "output": output_type
    })

claims = validated_claims

print("Structured claims:", len(claims))
print("Claim validation: PASSED")
print("Evidence keys:", list(evidence.keys()))


Structured claims: 42
Claim validation: PASSED
Evidence keys: ['image', 'clinical_note', 'retrieved']


In [5]:
# Sanity check before verification.
# This must show the same structured-claim count produced by Explanation Agent.

print("========================================")
print("VERIFIER INPUT SANITY CHECK")
print("========================================")
print("Claims received:", len(claims))
print("Expected claim field:", "claims" in explanation_output)
print("Evidence keys:", list(evidence.keys()))

if len(claims) == 0:
    raise ValueError(
        "Verifier received 0 claims. "
        "The Explanation Agent blackboard was loaded, but its structured "
        "'claims' list is empty."
    )

print("First claim:")
print(claims[0])
print("========================================")


VERIFIER INPUT SANITY CHECK
Claims received: 42
Expected claim field: True
Evidence keys: ['image', 'clinical_note', 'retrieved']
First claim:
{'claim': 'The model assessment is No DR (grade 0) with confidence 0.36.', 'output': 'clinician'}


In [6]:
def evidence_to_text(value):
    if value is None:
        return ""

    if isinstance(value, str):
        return value

    if isinstance(value, torch.Tensor):
        if value.numel() == 0:
            return ""
        return " ".join(
            map(str, value.detach().cpu().flatten().tolist())
        )

    if isinstance(value, dict):
        parts = []
        for key, val in value.items():
            text = evidence_to_text(val)
            if text:
                parts.append(f"{key}: {text}")
        return " ".join(parts)

    if isinstance(value, (list, tuple, set)):
        return " ".join(
            evidence_to_text(item)
            for item in value
            if evidence_to_text(item)
        )

    return str(value)


image_text = evidence_to_text(image_evidence)
clinical_text = evidence_to_text(clinical_evidence)
retrieved_text = evidence_to_text(retrieved_evidence)

evidence_sources = {
    "image_localization": image_text,
    "clinical_note": clinical_text,
    "retrieved_guidelines": retrieved_text,
}

print("Image evidence available:", bool(image_text.strip()))
print("Clinical-note evidence available:", bool(clinical_text.strip()))
print("Retrieved evidence available:", bool(retrieved_text.strip()))


Image evidence available: True
Clinical-note evidence available: True
Retrieved evidence available: True


In [7]:
# Lightweight MVP traceability check.
# This checks lexical provenance, not clinical truth and not full semantic entailment.

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "have", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "this", "to", "was", "were", "with", "which", "will",
    "may", "can", "could", "should", "would", "system", "current"
}

def normalize_tokens(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9%.-]+", " ", text)
    tokens = text.split()

    return {
        token.strip(".-")
        for token in tokens
        if token.strip(".-")
        and token.strip(".-") not in STOPWORDS
    }


def trace_claim(claim_text):
    claim_tokens = normalize_tokens(claim_text)

    if not claim_tokens:
        return {
            "status": "UNSUPPORTED",
            "evidence_source": None,
            "overlap": 0.0,
            "matched_tokens": []
        }

    best_source = None
    best_overlap = 0.0
    best_matches = []

    for source_name, source_text in evidence_sources.items():
        if not source_text:
            continue

        source_tokens = normalize_tokens(source_text)
        matches = sorted(claim_tokens & source_tokens)
        overlap = len(matches) / len(claim_tokens)

        if overlap > best_overlap:
            best_overlap = overlap
            best_source = source_name
            best_matches = matches

    supported = (
        best_overlap >= 0.40 and len(best_matches) >= 2
    ) or (
        best_overlap >= 0.75 and len(best_matches) >= 1
    )

    return {
        "status": "SUPPORTED" if supported else "UNSUPPORTED",
        "evidence_source": best_source,
        "overlap": round(best_overlap, 3),
        "matched_tokens": best_matches
    }


In [8]:
claim_results = []
supported_claims = []
dropped_claims = []
hallucination_events = []

for item in claims:
    claim_text = item["claim"]
    output_type = item.get("output", "unknown")

    trace = trace_claim(claim_text)

    result = {
        "claim": claim_text,
        "output": output_type,
        "status": trace["status"],
        "evidence_source": trace["evidence_source"],
        "overlap": trace["overlap"],
        "matched_tokens": trace["matched_tokens"]
    }

    claim_results.append(result)

    if trace["status"] == "SUPPORTED":
        supported_claims.append(result)
    else:
        dropped_claims.append(result)

        hallucination_events.append({
            "claim": claim_text,
            "output": output_type,
            "reason": (
                "Claim could not be traced to image localization L, "
                "clinical-note entities e, or retrieved evidence R."
            ),
            "evidence_source": trace["evidence_source"],
            "overlap": trace["overlap"]
        })

verified_clinician_claims = [
    x["claim"]
    for x in supported_claims
    if x["output"] == "clinician"
]

verified_patient_claims = [
    x["claim"]
    for x in supported_claims
    if x["output"] == "patient"
]

if len(dropped_claims) == 0:
    verification_status = "VERIFIED"
elif len(supported_claims) > 0:
    verification_status = "PARTIALLY_VERIFIED"
else:
    verification_status = "UNVERIFIED"

verification_output = {
    "agent": "VerifierAgent",
    "verification_status": verification_status,

    "clinician_explanation": " ".join(
        verified_clinician_claims
    ),

    "patient_explanation": " ".join(
        verified_patient_claims
    ),

    "claims_checked": len(claims),
    "supported_claims": len(supported_claims),
    "dropped_claims": dropped_claims,
    "hallucination_events": hallucination_events,

    "evidence_traceability": {
        "image_localization": bool(image_text.strip()),
        "clinical_note": bool(clinical_text.strip()),
        "retrieved_guidelines": bool(retrieved_text.strip())
    },

    "claim_results": claim_results,

    "review_required": len(dropped_claims) > 0,

    "verification_notes": [
        "Verifier checks claim traceability, not clinical truth.",
        "Claims are consumed directly from the Explanation Agent structured claims field.",
        "Unsupported claims are dropped and logged as hallucination events."
    ],

    "timestamp": datetime.now().isoformat()
}

torch.save(
    verification_output,
    VERIFIER_OUTPUT_PATH
)

print("Verifier Agent validation PASSED.")
print("Status:", verification_status)
print("Claims checked:", len(claims))
print("Supported:", len(supported_claims))
print("Dropped:", len(dropped_claims))
print("Saved:", VERIFIER_OUTPUT_PATH)


Verifier Agent validation PASSED.
Status: PARTIALLY_VERIFIED
Claims checked: 42
Supported: 32
Dropped: 10
Saved: /kaggle/working/verifier_blackboard.pt


In [9]:
print("DROPPED CLAIMS:")

for event in hallucination_events:
    print("-" * 80)
    print("CLAIM:", event["claim"])
    print("OUTPUT:", event["output"])
    print("SOURCE:", event["evidence_source"])
    print("OVERLAP:", event["overlap"])
    print("REASON:", event["reason"])


DROPPED CLAIMS:
--------------------------------------------------------------------------------
CLAIM: The model assessment is No DR (grade 0) with confidence 0.36.
OUTPUT: clinician
SOURCE: image_localization
OVERLAP: 0.375
REASON: Claim could not be traced to image localization L, clinical-note entities e, or retrieved evidence R.
--------------------------------------------------------------------------------
CLAIM: ...
OUTPUT: clinician
SOURCE: None
OVERLAP: 0.0
REASON: Claim could not be traced to image localization L, clinical-note entities e, or retrieved evidence R.
--------------------------------------------------------------------------------
CLAIM: ...
OUTPUT: clinician
SOURCE: None
OVERLAP: 0.0
REASON: Claim could not be traced to image localization L, clinical-note entities e, or retrieved evidence R.
--------------------------------------------------------------------------------
CLAIM: ...
OUTPUT: clinician
SOURCE: None
OVERLAP: 0.0
REASON: Claim could not be traced to

In [10]:
required_keys = [
    "agent",
    "verification_status",
    "clinician_explanation",
    "patient_explanation",
    "claims_checked",
    "supported_claims",
    "dropped_claims",
    "hallucination_events",
    "evidence_traceability",
    "claim_results",
    "review_required"
]

missing_keys = [
    key for key in required_keys
    if key not in verification_output
]

if missing_keys:
    raise ValueError(
        f"Verifier output missing keys: {missing_keys}"
    )

assert len(claims) > 0
assert verification_output["claims_checked"] == len(claims)
assert (
    verification_output["supported_claims"]
    + len(verification_output["dropped_claims"])
    == verification_output["claims_checked"]
)

print("Final Verifier validation: PASSED")
print("Blackboard ready:", VERIFIER_OUTPUT_PATH)


Final Verifier validation: PASSED
Blackboard ready: /kaggle/working/verifier_blackboard.pt
